#**TASK1**

In [1]:
!pip install datasets Huggingface_hub

In [2]:
from datasets import load_dataset
ds=load_dataset("roneneldan/TinyStories")

for i in range(3):
  print(f"Story-{i+1}")
  print(ds['train']['text'][i])
  print('-------------------------------------------------------------')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Story-1
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.
-------------------------------------------------------------
Story-2
Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.

One day, Beep was driving in 

**Next-Token Prediction**: Is the process by which autoregressive models generate text when prompted. In this process, the model when input with a prompt, breaks it into a set of tokens and analyses it to predict subsequent set of tokens based on pure mathematical probability one after the other.This keeps happening until the designated terminal point has been reached for generation.

Training the model on data is for it to understand grammar, sentence-structure rules etc so as to maximise the probability of predicting the correct subsequent word/token.

#**TASK2**

In [3]:
!pip install tiktoken numpy
import tiktoken

In [4]:
enc=tiktoken.get_encoding('gpt2')
input="Hi, I am Bhuvanesh"

token_ids=enc.encode(input)
print(f"Token ids: {token_ids}")
print(f"Total tokens:{len(token_ids)}")
dec_text=enc.decode(token_ids)
print(f'Decoded text:"{dec_text}"')

Token ids: [17250, 11, 314, 716, 347, 13415, 10438, 5069]
Total tokens:8
Decoded text:"Hi, I am Bhuvanesh"


In [5]:
import os
import numpy as np
from tqdm.auto import tqdm

def process(example):
  ids=enc.encode_ordinary(example['text'])
  out={'ids':ids,'len':len(ids)}
  return out

tokenized=ds.map(
    process,
    remove_columns=["text"],
    desc="Tokenizing splits",
    num_proc=2
)
data_dir="./data"
os.makedirs(data_dir,exist_ok=True)

for split,dset in tokenized.items():
  arr_len=np.sum(dset['len'],dtype=np.uint64)
  name='val'if split=='vaidation' else split
  filename=os.path.join(data_dir,f"{name}.bin")

  dtype=np.uint16
  arr=np.memmap(filename,dtype=dtype,mode="w+",shape=(arr_len,))
  total_batches=1024
  idx=0
  for batch_idx in tqdm(range(total_batches),desc=f"Writing {filename}"):
    batch=dset.shard(num_shards=total_batches,index=batch_idx,contiguous=True).with_format("numpy")
    arr_batch=np.concatenate(batch['ids'])
    arr[idx:idx+len(arr_batch)]=arr_batch
    idx+=len(arr_batch)
  arr.flush()
  print(f"saved {filename}")

Tokenizing splits (num_proc=2):   0%|          | 0/2119719 [00:00<?, ? examples/s]

Tokenizing splits (num_proc=2):   0%|          | 0/21990 [00:00<?, ? examples/s]

Writing ./data/train.bin:   0%|          | 0/1024 [00:00<?, ?it/s]

saved ./data/train.bin


Writing ./data/validation.bin:   0%|          | 0/1024 [00:00<?, ?it/s]

saved ./data/validation.bin


In [6]:
train_bin_path="./data/train.bin"
m=np.memmap(train_bin_path,dtype=np.uint16,mode='r')
first_50_tokens=m[:50]

print(f"Total number of tokens in train.bin:{len(m):,}")
print(f"First 50 tokens:{list(first_50_tokens)}")
print(f"Decoded text:\n{enc.decode(list(first_50_tokens))}")


Total number of tokens in train.bin:471,872,517
First 50 tokens:[np.uint16(3198), np.uint16(1110), np.uint16(11), np.uint16(257), np.uint16(1310), np.uint16(2576), np.uint16(3706), np.uint16(20037), np.uint16(1043), np.uint16(257), np.uint16(17598), np.uint16(287), np.uint16(607), np.uint16(2119), np.uint16(13), np.uint16(1375), np.uint16(2993), np.uint16(340), np.uint16(373), np.uint16(2408), np.uint16(284), np.uint16(711), np.uint16(351), np.uint16(340), np.uint16(780), np.uint16(340), np.uint16(373), np.uint16(7786), np.uint16(13), np.uint16(20037), np.uint16(2227), np.uint16(284), np.uint16(2648), np.uint16(262), np.uint16(17598), np.uint16(351), np.uint16(607), np.uint16(1995), np.uint16(11), np.uint16(523), np.uint16(673), np.uint16(714), np.uint16(34249), np.uint16(257), np.uint16(4936), np.uint16(319), np.uint16(607), np.uint16(10147), np.uint16(13), np.uint16(198)]
Decoded text:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play wit

In [7]:
import torch
device='cuda' if torch.cuda.is_available() else 'cpu'
batch_size=4
block_size=8

def get_batch(split):
  filename='./data/train.bin' if split=='train' else './data/val.bin'
  data=np.memmap(filename, dtype=np.uint16, mode='r')

  ix=torch.randint(len(data)-block_size,(batch_size,))
  x=torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
  y=torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])

  x,y=x.to(device), y.to(device)
  return x,y

x,y=get_batch('train')
x_sample=x[0].tolist()
y_sample=y[0].tolist()

print(f"x-shape:{x.shape}")
print(f"x sample:{x_sample}")
print(f"y-shape:{y.shape}")
print(f"y sample:{y_sample}")

shift_check=x_sample[1:]==y_sample[:-1]
print(f"{shift_check}")

x-shape:torch.Size([4, 8])
x sample:[379, 683, 13, 198, 198, 14967, 1820, 2936]
y-shape:torch.Size([4, 8])
y sample:[683, 13, 198, 198, 14967, 1820, 2936, 845]
True


In [35]:
import math
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

@dataclass
class GPTConfig:
  block_size:int=256
  vocab_size:int=50257
  n_layer:int=4
  n_head:int=4
  n_embed:int=128
  dropout:float=0.1
  bias:bool=True

In [36]:
class LayerNorm(nn.Module):
  def __init__(self,ndim,bias):
    super().__init__()
    self.weight=nn.Parameter(torch.ones(ndim))
    self.bias=nn.Parameter(torch.zeros(ndim)) if bias else None

  def forward(self,x):
    return F.layer_norm(x,self.weight.shape,self.weight,self.bias,1e-5)

In [52]:
class CausalSelfAttention(nn.Module):
  def __init__(self,config):
    super().__init__()
    assert config.n_embed % config.n_head==0
    self.c_attn=nn.Linear(config.n_embed, 3*config.n_embed)
    self.c_proj=nn.Linear(config.n_embed, config.n_embed)
    self.attn_dropout=nn.Dropout(config.dropout)
    self.resid_dropout=nn.Dropout(config.dropout)

    self.n_head=config.n_head
    self.n_embed=config.n_embed

    self.flash=hasattr(F,"scaled_dot_product_attention")
    if not self.flash:
      self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size)).view(1, 1, config.block_size, config.block_size))

  def forward(self,x):
    B,T,C=x.size()
    q,k,v=self.c_attn(x).split(self.n_embed, dim=2)

    k=k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
    q=q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
    v=v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)

    if self.flash:
      y=F.scaled_dot_product_attention(q,k,v,attn_mask=None,
                                       dropout_p=self.attn_dropout.p if self.training else 0.0,
                                       is_causal=True)
    else:
      att=(q@k.transpose(-2,-1))*(1.0/math.sqrt(k.size(-1)))
      att=att.masked_fill(self.bias[:,:,:T,:T]==0,float('-inf'))
      att=F.softmax(att,dim=-1)
      att=self.attn_dropout(att)
      y=att@v

    y=y.transpose(1,2).contiguous().view(B,T,C)
    y=self.resid_dropout(self.c_proj(y))
    return y

In [53]:
class MLP(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.c_fc= nn.Linear(config.n_embed, 4*config.n_embed)
    self.gelu=nn.GELU()
    self.c_proj=nn.Linear(4*config.n_embed, config.n_embed)
    self.dropout=nn.Dropout(config.dropout)

  def forward(self,x):
    x=self.c_fc(x)
    x=self.gelu(x)
    x=self.c_proj(x)
    x=self.dropout(x)
    return x

In [54]:
class Block(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.ln1=LayerNorm(config.n_embed, bias=config.bias)
    self.attn=CausalSelfAttention(config)
    self.ln2=LayerNorm(config.n_embed,bias=config.bias)
    self.mlp=MLP(config)

  def forward(self,x):
    x=x+self.attn(self.ln1(x))
    x=x+self.mlp(self.ln2(x))
    return x

In [55]:
class GPT(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.config=config

    self.transformer=nn.ModuleDict(dict(
        wte=nn.Embedding(config.vocab_size,config.n_embed),
        wpe=nn.Embedding(config.block_size,config.n_embed),
        drop=nn.Dropout(config.dropout),
        h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
        ln_f=LayerNorm(config.n_embed, bias=config.bias),
    ))
    self.lm_head=nn.Linear(config.n_embed,config.vocab_size,bias=False)
    self.transformer.wte.weight=self.lm_head.weight

    self.apply(self._init_weights)
    for pn, p in self.named_parameters():
      if pn.endswith('c_proj.weight'):
        nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2*config.n_layer))

  def _init_weights(self,module):
      if isinstance(module, nn.Linear):
          nn.init.normal_(module.weight, mean=0.0, std=0.02)
          if module.bias is not None:
              nn.init.zeros_(module.bias)
      elif isinstance(module, nn.Embedding):
          nn.init.normal_(module.weight, mean=0.0, std=0.02)

  def forward(self,idx,targets=None):
    device=idx.device
    b,t=idx.size()
    assert t<=self.config.block_size
    pos=torch.arange(0,t,dtype=torch.long,device=device)
    tok_emb=self.transformer.wte(idx)
    pos_emb=self.transformer.wpe(pos)
    x=self.transformer.drop(tok_emb+pos_emb)

    for block in self.transformer.h:
      x=block(x)
    x=self.transformer.ln_f(x)

    if targets is not None:
      logits=self.lm_head(x)
      loss=F.cross_entropy(logits.view(-1,logits.size(-1)),targets.view(-1),ignore_index=-1)
      return logits,loss
    else:
      logits=self.lm_head(x[:,[-1],:])
      return logits, None

  @torch.no_grad()
  def generate(self,idx,max_new_tokens,temperature=1.0,top_k=None):
    for _ in range(max_new_tokens):
      idx_cond=idx if idx.size(1)<=self.config.block_size else idx[:, -self.config.block_size:]
      logits,_=self(idx_cond)
      logits=logits[:,-1,:]/temperature

      if top_k is not None:
        v,_=torch.topk(logits,min(top_k,logits.size(-1)))
        logits[logits<v[:,[-1]]]=float('-inf')

      probs=F.softmax(logits,dim=-1)
      idx_next=torch.multinomial(probs,num_samples=1)
      idx=torch.cat((idx,idx_next),dim=1)
    return idx

In [56]:
config=GPTConfig(block_size=8)
model=GPT(config).to(device)

param_count=sum(p.numel() for p in model.parameters())
print(f"Total parameter capacity:{param_count}")

x,y=get_batch('train')
logits,loss=model(x,y)

print(f"Logits array dimn:{logits.shape}")
print(f"Loss score:{loss.item():.4f}")


Total parameter capacity:7227264
Logits array dimn:torch.Size([4, 8, 50257])
Loss score:10.8084


#**Task5**


In [60]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

def config_optim(model,weight_decay,learning_rate,betas):
  param_dict={pn:p for pn, p in model.named_parameters() if p.requires_grad}
  decay_params=[]
  nodecay_params=[]

  for mn, p in param_dict.items():
    if p.dim()>=2:
      decay_params.append(p)
    else:
      nodecay_params.append(p)

  optim_groups=[
      {'params':decay_params,'weight_decay':weight_decay},
      {'params':nodecay_params,'weight_decay':0.0}
  ]
  num_decay_vars=sum(p.numel() for p in decay_params)
  num_nodecay_vars=sum(p.numel() for p in nodecay_params)
  print(f"Num decay params:{num_decay_vars}")
  print(f"Num nodecay params:{num_nodecay_vars}")
  optimizer=optim.AdamW(optim_groups,lr=learning_rate,betas=betas)
  return optimizer

max_lr=6e-4
weight_decay=1e-1
betas=(0.9,0.95)
total_steps=2000

optimizer=config_optim(model,weight_decay, max_lr,betas)
scheduler=CosineAnnealingLR(optimizer,T_max=total_steps, eta_min=max_lr*0.1)

print("Optimizer configured")

Num decay params:7220352
Num nodecay params:6912
Optimizer configured


In [64]:
initial_lr=optimizer.param_groups[0]['lr']
print(f"Initial learning rate:{initial_lr:.10f}")

model.train()
optimizer.zero_grad()
x,y=get_batch('train')
logits,loss=model(x,y)

loss.backward()
optimizer.step()
scheduler.step()

updated_lr=optimizer.param_groups[0]['lr']
print(f"Updated learning rate:{updated_lr:.10f}")

is_sched_active=initial_lr!=updated_lr
print(f"Scheduler status:{is_sched_active}")

Initial learning rate:0.0005999970
Updated learning rate:0.0005999947
Scheduler status:True


#**Task6**

In [77]:
import torch
from contextlib import nullcontext

learning_rate = 1e-4
max_iters = 20000
warmup_steps = 1000
min_lr = 5e-4
eval_iters = 500
batch_size = 32
block_size = 128

gradient_accumulation_steps = 32

device =  "cuda" if torch.cuda.is_available() else "cpu"
device_type = 'cuda' if 'cuda' in device else 'cpu'
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]

ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

torch.set_default_device(device)
torch.manual_seed(42)

In [75]:
def estimate_loss(model):
    out = {}
    model.eval()
    with torch.inference_mode():
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                X, Y = get_batch(split)
                with ctx:
                    logits, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean()
    model.train()
    return out

In [81]:
if os.path.exists('./data/train.bin'):
    data_dir = './data'
elif os.path.exists('train.bin'):
    data_dir = '.'
else:
    data_dir = '.'

def get_batch(split):
    filename = os.path.join(data_dir, 'train.bin' if split == 'train' else 'val.bin')
    if not os.path.exists(filename):
        raise FileNotFoundError(f"Missing data file target at location: {filename}. Please re-run your Data Prep step!")
    data = np.memmap(filename, dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - 128, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+128]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+128]).astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

best_val_loss = float('inf')
best_model_params_path = "best_model_params.pt"
train_loss_list, validation_loss_list = [], []
model = model.to(device)
for epoch in tqdm(range(max_iters)):
    if epoch % eval_iters == 0 and epoch != 0:
        losses = estimate_loss(model)
        print(f"Epoch {epoch}: Train loss {losses['train']:.4f}, Val loss {losses['val']:.4f}")
        print(f"Learning rate: {optimizer.param_groups[0]['lr']:.5f}")
        train_loss_list += [losses['train']]
        validation_loss_list += [losses['val']]

        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save(model.state_dict(), best_model_params_path)

    x, y = get_batch("train")
    x, y = x.to(device), y.to(device)

    with ctx:
        logits, loss = model(x, y)
        loss = loss / gradient_accumulation_steps
    if scaler is not None:
        scaler.scale(loss).backward()
    else:
        loss.backward()

    if ((epoch + 1) % gradient_accumulation_steps == 0) or (epoch + 1 == max_iters):
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)

        if scaler is not None:
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()

        optimizer.zero_grad(set_to_none=True)
    scheduler.step()

  0%|          | 0/20000 [00:00<?, ?it/s]

/tmp/ipykernel_1890/1623887362.py:55: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


FileNotFoundError: Missing data file target at location: ./data/val.bin. Please re-run your Data Prep step!

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

iterations = [i * eval_iters for i in range(len(train_loss_list))]

if len(iterations) < len(train_loss_list):
    iterations.append(max_iters - 1)

plt.plot(iterations, train_loss_list, label='Train Loss', color='#1f77b4', linestyle='-', marker='o', markersize=4)
plt.plot(iterations, validation_loss_list, label='Val Loss', color='#ff7f0e', linestyle='--', marker='s', markersize=4)

plt.title('SLM Pre-training Loss Trajectory Graph', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Training Iterations / Epoch Steps', fontsize=12)
plt.ylabel('Cross-Entropy Metric Value', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6, color='gray')
plt.legend(fontsize=11, loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
model.eval()
with torch.inference_mode():
    final_losses = estimate_loss(model)
    val_loss = final_losses['val']
    perplexity = math.exp(val_loss)

print(f"{val_loss:.4f}")
print(f"{perplexity:.2f}")

temperatures = [0.2, 0.7, 1.2]
prompt_context = torch.tensor([[13]], dtype=torch.long, device=device)

for temp in temperatures:
    generated_idx = model.generate(prompt_context, max_new_tokens=50, temperature=temp, top_k=40)
    print(enc.decode(generated_idx[0].tolist()).strip())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
iterations = [i * eval_iters for i in range(len(train_loss_list))]
if len(iterations) < len(train_loss_list):
    iterations.append(max_iters - 1)

plt.plot(iterations, train_loss_list, label='Train')
plt.plot(iterations, validation_loss_list, label='Val')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
import torch

model.load_state_dict(torch.load("best_model_params.pt", map_location=device))
model.eval()

prompt = "Once upon a time"
context = torch.tensor([enc.encode(prompt)], dtype=torch.long, device=device)

temperatures = [0.2, 0.8]

with torch.inference_mode():
    for temp in temperatures:
        print(f"Temperature: {temp}")
        generated_idx = model.generate(context, max_new_tokens=120, temperature=temp, top_k=40)
        print(enc.decode(generated_idx[0].tolist()))